# 7) Reservoirs and pumped hydro: the value of water

## Objectives

At the end of this notebook you should be able to:

- model a **hydro reservoir** with natural inflow, a storage level and spillage — in your own Linopy model and as a PyPSA `StorageUnit`
- read the **water value** — the opportunity cost of stored water — as the dual of the reservoir's water balance
- show with the **KKT conditions** that a reservoir runs exactly in the hours when the price is at least the water value
- explain why a reservoir that offers its energy at zero cost, hour by hour, does the system a disservice
- add **pumped hydro**, and state when buying power to pump water uphill pays

Hydropower is the backbone of Croatia's electricity supply. Run-of-river plants (as in M2.2) produce what the river brings; **reservoir** plants store water and choose *when* to produce — and that choice is an optimisation problem. P2's error analysis warned that hydro is not a zero-cost plant; this lesson shows what it costs instead. One synthetic, Croatia-sized week; all numbers are **illustrative**.

In [1]:
import logging
import warnings

import linopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### A week with a reservoir

In [2]:
T = 168
h = np.arange(T)
hour, day = h % 24, h // 24
rng = np.random.default_rng(3)
load = (2000 + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0) + 750 * np.exp(-((hour - 19) / 2.0) ** 2)
        - 150 * (day >= 5) + rng.normal(0, 30, T))                                        # MW
cloud = np.repeat([0.9, 0.6, 1.0, 0.8, 0.4, 1.0, 0.9], 24)
solar = 2500 * 0.75 * np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * cloud
wind = 1000 * np.clip(0.3 + 0.25 * np.sin(2 * np.pi * h / 96), 0.02, 0.9)
free = solar + wind + 300.0                                  # solar + wind + run-of-river, all at 0 EUR/MWh
thermal = pd.DataFrame({"p_nom": [350, 900, 600, 800, 400], "mc": [12.0, 89.1, 94.2, 104.1, 128.9]},
                       index=pd.Index(["nuclear", "ccgt", "coal", "imports", "ocgt"], name="gen"))   # M2.2 at 70 EUR/t
RESERVOIR = {"p_nom": 1000, "e_nom": 30_000, "inflow": 120.0}     # MW turbine, MWh storage, MW average inflow
print(f"demand {load.min():,.0f}-{load.max():,.0f} MW; free energy {free.min():,.0f}-{free.max():,.0f} MW; "
      f"inflow {RESERVOIR['inflow'] * T:,.0f} MWh over the week")
thermal

demand 1,810-2,785 MW; free energy 350-2,638 MW; inflow 20,160 MWh over the week


         p_nom     mc
gen                  
nuclear    350   12.0
ccgt       900   89.1
coal       600   94.2
imports    800  104.1
ocgt       400  128.9

The reservoir has a 1,000 MW turbine but receives only 120 MW of water on average: it can produce at full power for only about 20 of the week's 168 hours. **When** it produces is the decision.

### The reservoir model

Three variables per hour: turbine output d[t] (MW), spillage s[t] (water released without producing), and the storage level e[t] (MWh, measured in the energy the water can still produce). The water balance:

**e[t] = e[t−1] + inflow[t] − d[t] − s[t]**,  0 ≤ e[t] ≤ e_max,  0 ≤ d[t] ≤ p_max

As in lesson 3, the level is cyclic: the reservoir ends the week as full as it started, so the model cannot cheat by emptying it.

In [3]:
def week(reservoir=True, pumped=None, e_nom=RESERVOIR["e_nom"], inflow=RESERVOIR["inflow"]):
    """Own Linopy model of the week. pumped: dict with p_nom, e_nom, eta (one-way efficiency) or None."""
    m = linopy.Model()
    t, G = pd.RangeIndex(T, name="t"), thermal.index
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(thermal.p_nom.to_numpy(), (T, 1)), coords=[t, G]), name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(free, coords=[t]), name="free")
    supply = g.sum("gen") + f
    if reservoir:
        d = m.add_variables(lower=0, upper=RESERVOIR["p_nom"], coords=[t], name="turbine")
        s = m.add_variables(lower=0, coords=[t], name="spill")
        e = m.add_variables(lower=0, upper=e_nom, coords=[t], name="level")
        m.add_constraints(e - e.roll(t=1) + d + s == inflow, name="water")
        supply = supply + d
    if pumped:
        c = m.add_variables(lower=0, upper=pumped["p_nom"], coords=[t], name="pump")
        q = m.add_variables(lower=0, upper=pumped["p_nom"], coords=[t], name="generate")
        u = m.add_variables(lower=0, upper=pumped["e_nom"], coords=[t], name="upper_basin")
        m.add_constraints(u - u.roll(t=1) - pumped["eta"] * c + q / pumped["eta"] == 0, name="pumped_balance")
        supply = supply + q - c
    m.add_constraints(supply == xr.DataArray(load, coords=[t]), name="balance")
    m.add_objective((g * xr.DataArray(thermal.mc.to_numpy(), coords=[G])).sum())
    m.solve(solver_name="highs", output_flag=False)
    return m


def pypsa_week(pumped=None, e_nom=RESERVOIR["e_nom"], inflow=RESERVOIR["inflow"]):
    n = pypsa.Network()
    n.set_snapshots(h)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, r in thermal.iterrows():
        n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc)
    n.add("StorageUnit", "reservoir", bus="HR", p_nom=RESERVOIR["p_nom"], max_hours=e_nom / RESERVOIR["p_nom"],
          p_min_pu=0.0, inflow=inflow, cyclic_state_of_charge=True)            # p_min_pu=0: it cannot pump
    if pumped:
        n.add("StorageUnit", "pumped", bus="HR", p_nom=pumped["p_nom"], max_hours=pumped["e_nom"] / pumped["p_nom"],
              efficiency_store=pumped["eta"], efficiency_dispatch=pumped["eta"], cyclic_state_of_charge=True)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n


m = week()
n = pypsa_week()
price = m.constraints["balance"].dual.to_numpy() + 0.0
assert np.isclose(m.objective.value, n.objective, rtol=1e-9)
assert np.abs(price - n.buses_t.marginal_price["HR"].to_numpy()).max() < 1e-6
turbine, level = m.variables["turbine"].solution.to_numpy(), m.variables["level"].solution.to_numpy()
print(f"own model = PyPSA: {m.objective.value:,.0f} EUR for the week, identical prices in all {T} hours")
print(f"turbine runs in {(turbine > 1e-6).sum()} hours, produces {turbine.sum():,.0f} MWh; "
      f"spilled {m.variables['spill'].solution.sum().item():,.0f} MWh; level peaks at {level.max():,.0f} MWh")
pd.Series(price).round(1).value_counts().sort_index().rename("hours at this price").to_frame().T

own model = PyPSA: 9,828,300 EUR for the week, identical prices in all 168 hours
turbine runs in 66 hours, produces 20,160 MWh; spilled 0 MWh; level peaks at 3,976 MWh


                     0.0   12.0  89.1  94.2
hours at this price    11    18    69    70

Your model and PyPSA's `StorageUnit` agree to the cent and in every hourly price. In PyPSA a reservoir is a `StorageUnit` with an `inflow` and `p_min_pu=0` (so it cannot pump); spillage is built in.

The reservoir uses all 20,160 MWh of the week's inflow, spills nothing, and runs in 66 of the 168 hours. Its level never rises above 3,976 MWh — about 13 % of the reservoir — so the size of the reservoir is not what limits it here; the water is. Four price levels appear: 0 €/MWh in 11 hours (free energy is curtailed), 12 €/MWh in 18 hours (nuclear is marginal), 89.1 €/MWh in 69 hours (the CCGT) and 94.2 €/MWh in 70 hours.

### The water value

The dual of the water balance is the **water value**: how much the week's cost would fall if one more MWh of water flowed into the reservoir. Linopy reports duals of equality constraints with a sign convention of its own, so look at its magnitude:

In [4]:
water_value = -m.constraints["water"].dual.to_numpy()
print("water value over the week (EUR/MWh):", np.unique(np.round(water_value, 3)))
wv = water_value[0]
running = (turbine > 1e-6) & (turbine < RESERVOIR["p_nom"] - 1e-6)
full = turbine >= RESERVOIR["p_nom"] - 1e-6
off = turbine <= 1e-6
assert np.allclose(price[running], wv), "turbine between its limits: price = water value"
assert (price[full] >= wv - 1e-6).all(), "turbine at full power: price >= water value"
assert (price[off] <= wv + 1e-6).all(), "turbine off: price <= water value"
print(f"KKT check: price = {wv:.1f} in all {running.sum()} hours with the turbine between its limits, "
      f">= {wv:.1f} in {full.sum()} hours at full power, <= {wv:.1f} in {off.sum()} hours when it is off")

water value over the week (EUR/MWh): [94.2]
KKT check: price = 94.2 in all 64 hours with the turbine between its limits, >= 94.2 in 2 hours at full power, <= 94.2 in 102 hours when it is off


The water value is **94.2 €/MWh in every hour** — exactly coal's marginal cost. The reservoir has too little water to replace all coal production, so the last MWh of water replaces coal, and that is what water is worth. It is the same in every hour because the level never reaches 0 or the reservoir's capacity: a MWh of water can be moved freely from any hour to any other, so it must be equally valuable everywhere. If the level did hit a limit, the water value would differ before and after that moment — the exercise shows it.

The KKT conditions of Maths-3 then dictate the schedule, and the check confirms them in all 168 hours: when the turbine runs between its limits (64 hours) the price equals the water value; at full power (2 hours) the price is at least the water value; when it is off (102 hours) the price is at most the water value. Read the other way round, this is how a hydro operator should bid: **at the water value, not at zero**. In the 64 hours in which the turbine runs between its limits, the reservoir itself is the marginal plant, setting the price at the value of its water.

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
axes[0].step(h, price, where="mid", lw=1.2, label="price")
axes[0].axhline(wv, color="tab:blue", ls="--", lw=1, label=f"water value ({wv:.1f} EUR/MWh)")
axes[0].set(ylabel="EUR/MWh", title="The reservoir produces exactly when the price reaches the water value")
axes[0].legend(fontsize=8, loc="lower left")
axes[1].fill_between(h, 0, turbine, step="mid", alpha=0.5, label="reservoir turbine (MW)")
axes[1].plot(h, level / 10, color="black", lw=1, label="storage level (MWh / 10)")
axes[1].set(xlabel="hour of the week", ylabel="MW  /  MWh/10")
axes[1].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

### Why a reservoir must not offer at zero

Suppose the reservoir is treated like run-of-river: it offers its water at 0 €/MWh in every hour, the market clears **one hour at a time**, and the reservoir produces as much as it is asked for until it is empty. Simulate that week:

In [6]:
def hour_by_hour_zero_offer(start_level):
    """Merit-order clearing hour by hour; the reservoir offers at 0 EUR/MWh until it runs dry."""
    level, cost = start_level, 0.0
    out, levels = np.zeros(T), np.zeros(T)
    for t in range(T):
        level += RESERVOIR["inflow"]
        residual = max(load[t] - free[t], 0.0)
        out[t] = min(RESERVOIR["p_nom"], level, residual)
        level -= out[t]
        levels[t] = level
        residual -= out[t]
        for gen, r in thermal.sort_values("mc").iterrows():
            take = min(r.p_nom, residual)
            cost += take * r.mc
            residual -= take
    return cost, out, levels


start = level[-1]                         # the optimal model's level at the start of the week (cyclic)
naive_cost, naive_out, naive_level = hour_by_hour_zero_offer(start)
empty_hours = int((naive_level < 1e-6).sum())
print(f"zero offer, hour by hour: {naive_cost:,.0f} EUR | optimal schedule: {m.objective.value:,.0f} EUR | "
      f"extra cost {naive_cost - m.objective.value:,.0f} EUR ({naive_cost / m.objective.value - 1:.1%})")
print(f"the naive reservoir produces {naive_out.sum():,.0f} MWh - {naive_out.sum() - turbine.sum():,.0f} MWh more than "
      f"the optimal one, because it ends the week emptier - and is empty in {empty_hours} of the {T} hours")

zero offer, hour by hour: 10,048,036 EUR | optimal schedule: 9,828,300 EUR | extra cost 219,736 EUR (2.2%)
the naive reservoir produces 21,219 MWh - 1,059 MWh more than the optimal one, because it ends the week emptier - and is empty in 143 of the 168 hours


The zero-offer reservoir does the wrong thing at every step. Offered at 0 €/MWh it is always dispatched first, so it runs whenever there is any residual demand — in hours when nuclear or the CCGT could have served cheaply — and is empty in 143 of the 168 hours. At the evening peaks, when its water would have replaced coal, imports and the OCGT, it has only the trickle of the hour's inflow left. The week costs 219,736 € more (2.2 %), even though the naive reservoir uses 1,059 MWh more water than the optimal one, because it ends the week emptier: the comparison is biased in its favour and it still loses.

This is the model error P2 warned about: a market model that treats reservoir hydro as a zero-cost plant puts its energy in the wrong hours and gets prices wrong in hydro-rich systems such as Croatia's. Real operators bid at their water value — estimated in practice from longer-term models of inflows and future prices.

### Pumped hydro

A **pumped-storage** plant has a lower and an upper basin. It pumps water up when power is cheap and lets it run down through the turbine when power is dear — a battery made of water, with a round-trip efficiency of about 75 %. Add one (illustrative: 300 MW, 8 hours, 0.866 efficiency each way):

In [7]:
PUMPED = {"p_nom": 300, "e_nom": 2400, "eta": 0.866}
mp = week(pumped=PUMPED)
np_ = pypsa_week(pumped=PUMPED)
assert np.isclose(mp.objective.value, np_.objective, rtol=1e-9)
assert np.abs(mp.constraints["balance"].dual.to_numpy() - np_.buses_t.marginal_price["HR"].to_numpy()).max() < 1e-6
pump, gen_ = mp.variables["pump"].solution.to_numpy(), mp.variables["generate"].solution.to_numpy()
p_with = mp.constraints["balance"].dual.to_numpy() + 0.0
pumping_price = float((p_with * pump).sum() / pump.sum())
generating_price = float((p_with * gen_).sum() / gen_.sum())
print(f"own model = PyPSA with pumped hydro: {mp.objective.value:,.0f} EUR (saves {m.objective.value - mp.objective.value:,.0f} EUR)")
print(f"pumped {pump.sum():,.0f} MWh at an average {pumping_price:.1f} EUR/MWh, "
      f"generated {gen_.sum():,.0f} MWh at an average {generating_price:.1f} EUR/MWh "
      f"(round trip {gen_.sum() / pump.sum():.3f} = 0.866 x 0.866)")
print("prices in the hours it pumps (EUR/MWh):", sorted(set(np.round(p_with[pump > 1e-6], 1).tolist())))
print("reservoir water value with pumped hydro (EUR/MWh):", np.unique(np.round(-mp.constraints["water"].dual.to_numpy(), 1)))
pd.Series(p_with).round(1).value_counts().sort_index().rename("hours at this price").to_frame().T

own model = PyPSA with pumped hydro: 9,423,145 EUR (saves 405,156 EUR)
pumped 6,568 MWh at an average 24.6 EUR/MWh, generated 4,926 MWh at an average 89.1 EUR/MWh (round trip 0.750 = 0.866 x 0.866)
prices in the hours it pumps (EUR/MWh): [0.0, 12.0, 66.8]
reservoir water value with pumped hydro (EUR/MWh): [89.1]


                     0.0   12.0  66.8  89.1
hours at this price     4    12    13   139

The pumped-storage plant saves 405,156 € in the week. It pumps 6,568 MWh at an average of 24.6 €/MWh and returns 4,926 MWh at 89.1 €/MWh — exactly 75 % of what it pumped. It pumps only in hours priced at 0, 12 and 66.8 €/MWh, and 66.8 is no accident: 0.75 × 89.1 = 66.8. Pumping one MWh returns 0.75 MWh worth 89.1 €/MWh, so pumping pays as long as **price when pumping ≤ round-trip efficiency × price when generating** — at 66.8 €/MWh it just breaks even, and the price settles there.

Notice what happened to the reservoir: its water value fell from 94.2 to **89.1 €/MWh**, and no hour is left at 94.2 €/MWh. The pumped storage now covers the hours in which coal was marginal, so the reservoir's last MWh of water only replaces the CCGT. Storage resources compete with each other: adding one lowers the value of the other — the cannibalisation of M2.2 and lesson 5, between storage plants instead of solar farms.

## Check your understanding

**The water value**

1. The water value here equals coal's marginal cost. Why exactly coal's — and not the CCGT's or the imports'? What would happen to the water value in a wet week with twice the inflow?
2. The level never comes close to the reservoir's 30,000 MWh. What would a much smaller reservoir change about the water value, and why?

**Pumped hydro**

1. Pumped storage loses a quarter of the energy it stores. Under what condition on the prices is one more round trip worth it?
2. Why is pumped hydro worth far more in this week than lesson 5's battery was in its year?